# 04: Imágenes y Dockerfiles: la receta de tu entorno

## Tu primer Dockerfile


# Archivo: Dockerfile (sin extensión, exactamente este nombre)
FROM python:3.12-slim

WORKDIR /app

COPY app.py .

CMD ["python", "app.py"]

## Las instrucciones fundamentales del Dockerfile
- **FROM:** "Empieza con esta imagen base". Siempre es la primera línea. Define los cimientos de tu imagen.
- **WORKDIR:** "Trabaja dentro de este directorio". Crea el directorio si no existe y se posiciona ahí. Es como hacer cd pero dentro de la imagen.
- **COPY:** "Copia archivos de tu ordenador al contenedor". COPY <origen-en-tu-PC> <destino-en-la-imagen>.
- **RUN:** "Ejecuta este comando DURANTE la construcción de la imagen". Para instalar dependencias, crear directorios, etc.
- **CMD:** "Ejecuta este comando CUANDO el contenedor arranque". Es el comando por defecto al hacer docker run.
- **ENV:** "Define una variable de entorno". Disponible tanto durante la build como en runtime.
- **EXPOSE:** "Documenta que este contenedor escucha en este puerto". Es informativo — no abre el puerto realmente.


## docker build: construir tu imagen


1. Construir la imagen (el punto "." indica el directorio actual)
    - docker build -t mi-app:1.0 .
2.  Desglose:
    - docker build  → comando para construir
    - -t mi-app:1.0 → nombre:tag de tu imagen
    - .             → directorio donde está el Dockerfile (contexto de build)
3.  Verás algo como:
    - [1/3] FROM python:3.12-slim
    - [2/3] WORKDIR /app
    - [3/3] COPY app.py .
    - => naming to docker.io/library/mi-app:1.0
4. Verificar que la imagen existe
    - docker images
5. Ejecutar tu imagen personalizada
    - docker run mi-app:1.0

**Consejo de senior:** 

- SIEMPRE versiona tus imágenes con tags numéricos (mi-app:1.0, mi-app:1.1, mi-app:2.0).
-  Nunca uses solo "latest". Cuando algo se rompe en producción y necesitas volver a la versión anterior, agradecer tener tags es lo primero que harás.



## Un Dockerfile más realista: con dependencias


**Archivo: requirements.txt**

requests==2.31.0
pandas==2.1.4

¿Por qué copiamos requirements.txt ANTES que el resto del código? 

- Por el CACHE de capas. 
- Si cambias tu app.py pero no tocas requirements.txt, Docker reutiliza la capa de pip install del cache y solo recopia tu código. 
- Esto ahorra MINUTOS en cada build. 
  - Si copiaras todo junto (COPY . .), cualquier cambio en cualquier archivo invalidaría el cache del pip install.



El orden de las instrucciones en un Dockerfile IMPORTA MUCHO para el rendimiento.

- Regla de oro: 
  - pon las cosas que cambian MENOS frecuentemente arriba (instalar dependencias del sistema, pip install) 
  -  las que cambian MÁS frecuentemente abajo (copiar tu código). 

Así maximizas el cache.


## .dockerignore: lo que NO quieres copiar


Igual que .gitignore le dice a Git qué archivos ignorar, .dockerignore le dice a Docker qué NO copiar al contexto de build. 

Esto hace la build más rápida y tu imagen más pequeña:



**Consejo de senior:**

- si tu build es lenta, revisa el .dockerignore.
-  He visto builds que tardaban 5 minutos porque copiaban la carpeta .git (con todo el historial), node_modules, o datasets de prueba de varios GB. 
-  El .dockerignore es la primera línea de defensa contra builds lentas.

